# Final Anatomical Variation Score Refit After Freezing the Split

This notebook **freezes the existing patient-level split** and then **rebuilds the anatomical variation score using development-set-only fitting**.

## Why this notebook exists

The previous variation score was useful for creating a balanced provisional split, but its robust normalization and stratum thresholds were estimated using all cases. For a stricter evaluation workflow, the split must be frozen first, and the final variation score must then be refit using **development cases only** and applied unchanged to the holdout test cases.

## Method decisions used here

1. **Case-level grouping is preserved.** Repeated scans from the same case stay in the same split.
2. **The holdout test set is frozen.** We do not change which cases are in development versus test.
3. **Robust standardization is fit on development cases only.** The fitted parameters are then applied to both development and test.
4. **Primary score excludes duodenum.** Duodenum remains available only for sensitivity analysis because its pseudo-labels were less reliable in QC.
5. **The current feature set remains provisional.** It uses QC-filtered organ volume, slice span, and bounding-box extents. This is acceptable for a split-balanced engineering score, but a later upgrade to IBSI/PyRadiomics shape features would strengthen the final scientific version.

## What this notebook saves

- refit variation features
- robust fitting parameters
- refit scan-level and case-level variation scores
- development-only score thresholds
- refit holdout and fold balance tables
- updated split/manifests with the refit final score

In [ ]:
from pathlib import Path
import sys
import json

def find_project_root(start_path=None):
    if start_path is None:
        start_path = Path.cwd()
    start_path = Path(start_path).resolve()
    for path in [start_path] + list(start_path.parents):
        if (path / "src").exists():
            return path
    raise FileNotFoundError("Could not find project root containing src/")

project_root = find_project_root()

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Current working directory:", Path.cwd())
print("Project root:", project_root)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold

## Load the frozen split, organ table, and provisional variation outputs

The split is treated as fixed input. This notebook does **not** reshuffle or reassign cases between development and test.

In [ ]:
split_cases_path = project_root / "splits" / "patient_level_holdout_test.csv"
fold_assignments_path = project_root / "splits" / "fold_assignments_case_level.csv"
organ_long_path = project_root / "metadata" / "eda" / "final_mask_level_totalseg" / "organ_level_long_qc.csv"
provisional_case_scores_path = project_root / "metadata" / "eda" / "final_variation_score" / "variation_scores_per_case.csv"

required_paths = [
    split_cases_path,
    fold_assignments_path,
    organ_long_path,
]

for p in required_paths:
    if not p.exists():
        raise FileNotFoundError(f"Missing required file: {p}")

split_cases_frozen_df = pd.read_csv(split_cases_path)
fold_assignments_df = pd.read_csv(fold_assignments_path)
organ_long_df = pd.read_csv(organ_long_path)

if provisional_case_scores_path.exists():
    provisional_case_scores_df = pd.read_csv(provisional_case_scores_path)
else:
    provisional_case_scores_df = None

print("Frozen split rows:", len(split_cases_frozen_df))
print("Fold assignment rows:", len(fold_assignments_df))
print("Organ-level rows:", len(organ_long_df))
print("Has provisional case score file:", provisional_case_scores_df is not None)

split_cases_frozen_df.head()

## Define organs and features

### Primary score organs
- spleen
- kidney_left
- kidney_right
- stomach
- colon
- small_bowel

### Sensitivity score organs
- primary score organs plus duodenum

### Provisional engineering features used here
- `volume_ml_qc`
- `n_slices_present`
- `bbox_size_x`
- `bbox_size_y`
- `bbox_size_z`

These are acceptable for the current split-balanced engineering score. They should later be upgraded to more standardized shape features if you want the strongest final scientific version.

In [ ]:
PRIMARY_ORGANS = [
    "spleen",
    "kidney_left",
    "kidney_right",
    "stomach",
    "colon",
    "small_bowel",
]

SENSITIVITY_ORGANS = PRIMARY_ORGANS + ["duodenum"]

FEATURE_COLUMNS = [
    "volume_ml_qc",
    "n_slices_present",
    "bbox_size_x",
    "bbox_size_y",
    "bbox_size_z",
]

print("Primary organs:", PRIMARY_ORGANS)
print("Sensitivity organs:", SENSITIVITY_ORGANS)
print("Feature columns:", FEATURE_COLUMNS)

## Build the scan-level variation feature table

We keep only rows that passed QC for score construction.

In [ ]:
def parse_case_id(file_name: str) -> str:
    # Expected pattern: case101_day20.nii.gz -> case101
    return str(file_name).split("_day")[0]

variation_source_df = organ_long_df.copy()
variation_source_df["case_id"] = variation_source_df["file_name"].apply(parse_case_id)

# Only keep organs used in either primary or sensitivity score
variation_source_df = variation_source_df[
    variation_source_df["organ_name"].isin(SENSITIVITY_ORGANS)
].copy()

# Only keep QC-usable rows for score construction
variation_source_df = variation_source_df[
    variation_source_df["use_for_volume_stats"] == True
].copy()

missing_feature_cols = [col for col in FEATURE_COLUMNS if col not in variation_source_df.columns]
if missing_feature_cols:
    raise ValueError(f"Missing feature columns in organ_long_df: {missing_feature_cols}")

variation_features_long_df = variation_source_df.melt(
    id_vars=["file_name", "case_id", "organ_name", "present", "qc_flag", "use_for_volume_stats"],
    value_vars=FEATURE_COLUMNS,
    var_name="feature_name",
    value_name="feature_value",
)

variation_features_long_df = variation_features_long_df.dropna(subset=["feature_value"]).copy()

# Merge frozen split
split_cols = ["case_id", "split"]
if "n_scans" in split_cases_frozen_df.columns:
    split_cols.append("n_scans")

variation_features_long_df = variation_features_long_df.merge(
    split_cases_frozen_df[split_cols].drop_duplicates(),
    on="case_id",
    how="left",
)

if variation_features_long_df["split"].isna().any():
    missing_cases = sorted(variation_features_long_df.loc[variation_features_long_df["split"].isna(), "case_id"].unique())
    raise ValueError(f"Some cases are missing split assignment: {missing_cases[:10]}")

print("Feature rows used for refit:", len(variation_features_long_df))
variation_features_long_df.head()

## Fit robust standardization on development cases only

For each organ-feature pair:
- center = development-set median
- scale = development-set MAD
- if MAD is zero or invalid, fall back to IQR-based scale
- if that is also zero, use 1.0 to avoid division by zero

The fitted parameters are then applied unchanged to both development and test.

In [ ]:
def robust_fit_mad(values: pd.Series):
    values = pd.Series(values).dropna().astype(float)
    if len(values) == 0:
        return np.nan, np.nan

    center = float(values.median())
    mad = float(np.median(np.abs(values - center)))

    if np.isnan(mad) or mad == 0.0:
        q1 = float(values.quantile(0.25))
        q3 = float(values.quantile(0.75))
        iqr = q3 - q1
        # approximate robust scale from IQR
        scale = iqr / 1.349 if iqr > 0 else 1.0
    else:
        # 1.4826 * MAD approximates standard deviation under normality
        scale = 1.4826 * mad

    if np.isnan(scale) or scale == 0.0:
        scale = 1.0

    return center, float(scale)

fit_rows = []
dev_feature_df = variation_features_long_df[variation_features_long_df["split"] == "dev"].copy()

for (organ_name, feature_name), g in dev_feature_df.groupby(["organ_name", "feature_name"]):
    center, scale = robust_fit_mad(g["feature_value"])
    fit_rows.append(
        {
            "organ_name": organ_name,
            "feature_name": feature_name,
            "fit_center_dev_only": center,
            "fit_scale_dev_only": scale,
            "n_dev_feature_rows": len(g),
        }
    )

fit_params_df = pd.DataFrame(fit_rows).sort_values(["organ_name", "feature_name"]).reset_index(drop=True)
fit_params_df

In [ ]:
variation_features_refit_df = variation_features_long_df.merge(
    fit_params_df,
    on=["organ_name", "feature_name"],
    how="left",
)

variation_features_refit_df["robust_z_devfit"] = (
    (variation_features_refit_df["feature_value"] - variation_features_refit_df["fit_center_dev_only"])
    / variation_features_refit_df["fit_scale_dev_only"]
)

variation_features_refit_df["abs_robust_z_devfit"] = variation_features_refit_df["robust_z_devfit"].abs()

variation_features_refit_df.head()

## Build refit scan-level primary and sensitivity scores

In [ ]:
primary_long_df = variation_features_refit_df[
    variation_features_refit_df["organ_name"].isin(PRIMARY_ORGANS)
].copy()

sensitivity_long_df = variation_features_refit_df[
    variation_features_refit_df["organ_name"].isin(SENSITIVITY_ORGANS)
].copy()

scan_primary_scores_df = (
    primary_long_df
    .groupby(["file_name", "case_id", "split"])["abs_robust_z_devfit"]
    .mean()
    .reset_index(name="primary_variation_score_refit")
)

scan_sensitivity_scores_df = (
    sensitivity_long_df
    .groupby(["file_name", "case_id", "split"])["abs_robust_z_devfit"]
    .mean()
    .reset_index(name="sensitivity_variation_score_refit")
)

scan_variation_scores_refit_df = scan_primary_scores_df.merge(
    scan_sensitivity_scores_df,
    on=["file_name", "case_id", "split"],
    how="outer",
)

scan_variation_scores_refit_df = scan_variation_scores_refit_df.sort_values(
    ["case_id", "file_name"]
).reset_index(drop=True)

scan_variation_scores_refit_df.head()

## Aggregate to case level using the median across repeated scans

In [ ]:
case_primary_scores_df = (
    scan_variation_scores_refit_df
    .groupby(["case_id", "split"])["primary_variation_score_refit"]
    .median()
    .reset_index()
)

case_sensitivity_scores_df = (
    scan_variation_scores_refit_df
    .groupby(["case_id", "split"])["sensitivity_variation_score_refit"]
    .median()
    .reset_index()
)

case_n_scans_df = (
    scan_variation_scores_refit_df
    .groupby(["case_id", "split"])["file_name"]
    .nunique()
    .reset_index(name="n_scans")
)

case_variation_scores_refit_df = case_primary_scores_df.merge(
    case_sensitivity_scores_df,
    on=["case_id", "split"],
    how="inner",
).merge(
    case_n_scans_df,
    on=["case_id", "split"],
    how="left",
).sort_values("primary_variation_score_refit", ascending=False).reset_index(drop=True)

case_variation_scores_refit_df.head()

## Freeze final variation thresholds using development cases only

The final thresholds are estimated on development cases only and then applied unchanged to development and test.

In [ ]:
def compute_tertile_thresholds_dev_only(series: pd.Series):
    series = pd.Series(series).dropna().astype(float)
    q1 = float(series.quantile(1/3))
    q2 = float(series.quantile(2/3))
    return q1, q2

def apply_thresholds_to_strata(series: pd.Series, q1: float, q2: float):
    return pd.cut(
        series,
        bins=[-np.inf, q1, q2, np.inf],
        labels=["low", "medium", "high"],
        include_lowest=True,
    ).astype("object")

dev_case_scores_refit_df = case_variation_scores_refit_df[
    case_variation_scores_refit_df["split"] == "dev"
].copy()

primary_q1, primary_q2 = compute_tertile_thresholds_dev_only(
    dev_case_scores_refit_df["primary_variation_score_refit"]
)

sensitivity_q1, sensitivity_q2 = compute_tertile_thresholds_dev_only(
    dev_case_scores_refit_df["sensitivity_variation_score_refit"]
)

case_variation_scores_refit_df["primary_variation_stratum_final"] = apply_thresholds_to_strata(
    case_variation_scores_refit_df["primary_variation_score_refit"],
    primary_q1,
    primary_q2,
)

case_variation_scores_refit_df["sensitivity_variation_stratum_final"] = apply_thresholds_to_strata(
    case_variation_scores_refit_df["sensitivity_variation_score_refit"],
    sensitivity_q1,
    sensitivity_q2,
)

scan_variation_scores_refit_df = scan_variation_scores_refit_df.merge(
    case_variation_scores_refit_df[
        [
            "case_id",
            "primary_variation_stratum_final",
            "sensitivity_variation_stratum_final",
        ]
    ],
    on="case_id",
    how="left",
)

thresholds_df = pd.DataFrame(
    [
        {
            "score_name": "primary_variation_score_refit",
            "fit_split": "dev_only",
            "q1_threshold": primary_q1,
            "q2_threshold": primary_q2,
        },
        {
            "score_name": "sensitivity_variation_score_refit",
            "fit_split": "dev_only",
            "q1_threshold": sensitivity_q1,
            "q2_threshold": sensitivity_q2,
        },
    ]
)

thresholds_df

## Refit holdout balance checks

In [ ]:
holdout_balance_refit_df = (
    case_variation_scores_refit_df
    .groupby(["split", "primary_variation_stratum_final"])
    .size()
    .reset_index(name="n_cases")
    .sort_values(["split", "primary_variation_stratum_final"])
)

holdout_score_summary_refit_df = (
    case_variation_scores_refit_df
    .groupby("split")["primary_variation_score_refit"]
    .agg(["count", "mean", "median", "min", "max", "std"])
    .reset_index()
)

holdout_balance_refit_df, holdout_score_summary_refit_df

In [ ]:
plt.figure(figsize=(7, 4))
plot_df = holdout_balance_refit_df.pivot(
    index="primary_variation_stratum_final",
    columns="split",
    values="n_cases",
).fillna(0)
plot_df.plot(kind="bar", rot=0)
plt.ylabel("Number of cases")
plt.title("Refit holdout balance by final variation stratum")
plt.tight_layout()
plt.show()

## Refit fold balance checks

Fold membership is frozen. Only the score and stratum labels are updated.

In [ ]:
dev_case_final_df = case_variation_scores_refit_df[
    case_variation_scores_refit_df["split"] == "dev"
][["case_id", "primary_variation_score_refit", "primary_variation_stratum_final"]].copy()

fold_balance_refit_df = (
    fold_assignments_df
    .merge(dev_case_final_df, on="case_id", how="left")
    .groupby(["fold", "fold_role", "primary_variation_stratum_final"])
    .size()
    .reset_index(name="n_cases")
    .sort_values(["fold", "fold_role", "primary_variation_stratum_final"])
)

fold_score_summary_refit_df = (
    fold_assignments_df
    .merge(dev_case_final_df, on="case_id", how="left")
    .groupby(["fold", "fold_role"])["primary_variation_score_refit"]
    .agg(["count", "mean", "median", "min", "max", "std"])
    .reset_index()
)

fold_balance_refit_df.head(20), fold_score_summary_refit_df.head(10)

In [ ]:
N_FOLDS = int(fold_assignments_df["fold"].max() + 1)

fig, axes = plt.subplots(nrows=N_FOLDS, ncols=1, figsize=(8, 3 * N_FOLDS), sharex=True)

for fold_id in range(N_FOLDS):
    ax = axes[fold_id]
    plot_df = fold_balance_refit_df[fold_balance_refit_df["fold"] == fold_id].pivot(
        index="primary_variation_stratum_final",
        columns="fold_role",
        values="n_cases",
    ).fillna(0)
    plot_df.plot(kind="bar", ax=ax, rot=0)
    ax.set_title(f"Fold {fold_id} balance by final variation stratum")
    ax.set_ylabel("Number of cases")

plt.tight_layout()
plt.show()

## Compare provisional versus refit case-level scores

This lets us see how much the development-only refit changed rankings.

In [ ]:
if provisional_case_scores_df is not None:
    compare_cols = [
        "case_id",
        "primary_variation_score",
        "primary_variation_stratum",
        "sensitivity_variation_score",
        "sensitivity_variation_stratum",
    ]
    provisional_small_df = provisional_case_scores_df[compare_cols].copy()

    provisional_small_df["primary_rank_provisional"] = provisional_small_df["primary_variation_score"].rank(
        ascending=False, method="min"
    )
    provisional_small_df["sensitivity_rank_provisional"] = provisional_small_df["sensitivity_variation_score"].rank(
        ascending=False, method="min"
    )

    case_compare_df = case_variation_scores_refit_df.merge(
        provisional_small_df,
        on="case_id",
        how="left",
    )

    case_compare_df["primary_rank_refit"] = case_compare_df["primary_variation_score_refit"].rank(
        ascending=False, method="min"
    )
    case_compare_df["sensitivity_rank_refit"] = case_compare_df["sensitivity_variation_score_refit"].rank(
        ascending=False, method="min"
    )

    case_compare_df["primary_rank_shift_abs"] = (
        case_compare_df["primary_rank_refit"] - case_compare_df["primary_rank_provisional"]
    ).abs()

    case_compare_df["sensitivity_rank_shift_abs"] = (
        case_compare_df["sensitivity_rank_refit"] - case_compare_df["sensitivity_rank_provisional"]
    ).abs()

    case_compare_df = case_compare_df.sort_values(
        "primary_rank_shift_abs", ascending=False
    ).reset_index(drop=True)

    case_compare_df.head(20)
else:
    case_compare_df = pd.DataFrame()
    print("No provisional case score file found. Skipping provisional-vs-refit comparison.")

## Updated manifests with the refit final variation score

We keep the frozen split membership and add the final refit score/stratum columns.

In [ ]:
# Case-level split table with refit scores
split_cases_refit_df = split_cases_frozen_df.drop(
    columns=[c for c in split_cases_frozen_df.columns if "variation" in c or c == "n_scans"],
    errors="ignore",
).merge(
    case_variation_scores_refit_df[
        [
            "case_id",
            "split",
            "n_scans",
            "primary_variation_score_refit",
            "primary_variation_stratum_final",
            "sensitivity_variation_score_refit",
            "sensitivity_variation_stratum_final",
        ]
    ],
    on=["case_id", "split"],
    how="left",
).sort_values("case_id").reset_index(drop=True)

# Scan-level manifest
scan_manifest_refit_df = (
    organ_long_df[["file_name"]]
    .drop_duplicates()
    .copy()
)
scan_manifest_refit_df["case_id"] = scan_manifest_refit_df["file_name"].apply(parse_case_id)

scan_manifest_refit_df = scan_manifest_refit_df.merge(
    split_cases_refit_df[
        [
            "case_id",
            "split",
            "n_scans",
            "primary_variation_score_refit",
            "primary_variation_stratum_final",
            "sensitivity_variation_score_refit",
            "sensitivity_variation_stratum_final",
        ]
    ],
    on="case_id",
    how="left",
)

scan_manifest_refit_df["scan_path"] = scan_manifest_refit_df["file_name"].apply(
    lambda x: f"{{public_root}}/{x}"
)
scan_manifest_refit_df["mask_path"] = scan_manifest_refit_df["file_name"].apply(
    lambda x: f"{{totalseg_root}}/{x}"
)

scan_manifest_refit_df = scan_manifest_refit_df.sort_values(
    ["case_id", "file_name"]
).reset_index(drop=True)

# Scan-level fold manifest
scan_fold_refit_rows = []
for fold_id, fold_subset in fold_assignments_df.groupby("fold"):
    fold_case_roles = fold_subset[["case_id", "fold_role"]].copy()
    fold_scans = scan_manifest_refit_df[scan_manifest_refit_df["split"] == "dev"].copy()
    fold_scans = fold_scans.merge(fold_case_roles, on="case_id", how="left")
    fold_scans["fold"] = fold_id
    if fold_scans["fold_role"].isna().any():
        raise ValueError(f"Unknown fold assignments found in fold {fold_id}")
    scan_fold_refit_rows.append(fold_scans)

scan_fold_manifest_refit_df = pd.concat(scan_fold_refit_rows, ignore_index=True)

print("Refit case rows:", len(split_cases_refit_df))
print("Refit scan rows:", len(scan_manifest_refit_df))
split_cases_refit_df.head()

## Save outputs

In [ ]:
output_dir = project_root / "metadata" / "eda" / "final_variation_score_refit"
output_dir.mkdir(parents=True, exist_ok=True)

figure_dir = project_root / "docs" / "figures" / "final_variation_score_refit"
figure_dir.mkdir(parents=True, exist_ok=True)

split_dir = project_root / "splits"
model_dir = project_root / "metadata" / "modeling"

fit_params_df.to_csv(output_dir / "robust_fit_params_dev_only.csv", index=False)
variation_features_refit_df.to_csv(output_dir / "variation_features_long_refit.csv", index=False)
scan_variation_scores_refit_df.to_csv(output_dir / "variation_scores_per_scan_refit.csv", index=False)
case_variation_scores_refit_df.to_csv(output_dir / "variation_scores_per_case_refit.csv", index=False)
thresholds_df.to_csv(output_dir / "variation_thresholds_dev_only.csv", index=False)
holdout_balance_refit_df.to_csv(output_dir / "holdout_balance_refit.csv", index=False)
holdout_score_summary_refit_df.to_csv(output_dir / "holdout_score_summary_refit.csv", index=False)
fold_balance_refit_df.to_csv(output_dir / "fold_balance_refit.csv", index=False)
fold_score_summary_refit_df.to_csv(output_dir / "fold_score_summary_refit.csv", index=False)

if len(case_compare_df) > 0:
    case_compare_df.to_csv(output_dir / "provisional_vs_refit_case_comparison.csv", index=False)

split_cases_refit_df.to_csv(split_dir / "patient_level_holdout_test_refit_scored.csv", index=False)
split_cases_refit_df[split_cases_refit_df["split"] == "dev"].to_csv(
    split_dir / "dev_case_table_refit_scored.csv",
    index=False,
)
scan_manifest_refit_df.to_csv(model_dir / "modeling_manifest_refit_scored.csv", index=False)
scan_fold_manifest_refit_df.to_csv(model_dir / "scan_fold_manifest_refit_scored.csv", index=False)

# Save figures
plt.figure(figsize=(8, 4))
plt.hist(case_variation_scores_refit_df["primary_variation_score_refit"], bins=20)
plt.xlabel("Primary variation score (refit)")
plt.ylabel("Number of cases")
plt.title("Refit case-level primary variation score distribution")
plt.tight_layout()
plt.savefig(figure_dir / "refit_primary_variation_score_distribution.png", dpi=150)
plt.close()

plt.figure(figsize=(7, 4))
plot_df = holdout_balance_refit_df.pivot(
    index="primary_variation_stratum_final",
    columns="split",
    values="n_cases",
).fillna(0)
plot_df.plot(kind="bar", rot=0)
plt.ylabel("Number of cases")
plt.title("Refit holdout balance by final variation stratum")
plt.tight_layout()
plt.savefig(figure_dir / "refit_holdout_balance_by_stratum.png", dpi=150)
plt.close()

organ_contribution_refit_df = (
    primary_long_df
    .groupby("organ_name")["abs_robust_z_devfit"]
    .mean()
    .reset_index(name="mean_abs_robust_z_devfit")
    .sort_values("mean_abs_robust_z_devfit", ascending=False)
)

organ_contribution_refit_df.to_csv(output_dir / "organ_contribution_refit.csv", index=False)

plt.figure(figsize=(8, 4))
plt.bar(organ_contribution_refit_df["organ_name"], organ_contribution_refit_df["mean_abs_robust_z_devfit"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Mean absolute robust z")
plt.title("Refit organ contribution to primary variation score")
plt.tight_layout()
plt.savefig(figure_dir / "refit_organ_contribution_barplot.png", dpi=150)
plt.close()

print("Saved refit variation outputs to:", output_dir)
print("Saved refit figures to:", figure_dir)
print("Saved refit split tables to:", split_dir)
print("Saved refit modeling manifests to:", model_dir)

## Final interpretation template

Use wording close to this in the thesis/report:

The patient-level split was frozen first, after which the anatomical variation score was recomputed using development-set-only fitting. Robust standardization parameters and stratum thresholds were estimated from development cases only and then applied unchanged to the holdout test set. This reduces information leakage from the test set into the final evaluation-time anatomical variation definition. The primary score excluded duodenum because duodenum pseudo-labels were less reliable during mask QC; a sensitivity score including duodenum was retained separately.